In [ ]:
from agents.mcp import MCPServerStdio
from agents import Agent, Runner, trace
from IPython.display import display, Markdown
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client
import mcp
from openai import OpenAI
import json

In [ ]:
params = {"command": "uv", "args": ["run", "date-server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
  mcp_tools = await server.list_tools()

In [ ]:
mcp_tools

In [ ]:
instructions = "You are able to tell what today's date is"
request = "What is today's date?"
model = "gpt-4.1-nano"

In [ ]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
  agent = Agent(name="date_agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
  with trace("date_agent"):
    result = await Runner.run(agent, request)
  display(Markdown(result.final_output))

In [ ]:
server_params = StdioServerParameters(command="uv", args=["run", "date-server.py"], env=None)

async def call_tool(tool_name):
  async with stdio_client(server_params) as streams:
    async with mcp.ClientSession(*streams) as session:
      await session.initialize()
      result = await session.call_tool(tool_name)
      return result

async def get_tools():
  async with stdio_client(server_params) as streams:
    async with mcp.ClientSession(*streams) as session:
      await session.initialize()
      tools_result = await session.list_tools()
      openai_tools = []
      for tool in tools_result.tools:
        data = tool.model_dump(mode="json", exclude_none=True)
        name = data["name"]
        description = data.get("description")
        parameters = data.get("inputSchema")

        openai_tool = {
          "type": "function",
          "function": {
            "name": name,
            "description": description,
            "parameters": parameters
          }
        }
        openai_tools.append(openai_tool)
      return openai_tools

tools = await get_tools()

In [ ]:
print(tools)

In [ ]:
openai = OpenAI()

MODEL = "gpt-4.1-nano"
messages = [
  {"role": "system", "content": instructions},
  {"role": "user", "content": request}
]

async def handle_tool_calls(message):
  responses = []
  for tool_call in message.tool_calls:
    result = await call_tool(tool_call.function.name)

    responses.append({
      "role": "tool",
      "content": result.content,
      "tool_call_id": tool_call.id
    })
  return responses

response = openai.chat.completions.create(
  model=MODEL,
  messages=messages,
  tools=tools
)

while response.choices[0].finish_reason=="tool_calls":
  message = response.choices[0].message
  responses = await handle_tool_calls(message)
  messages.append(message)
  messages.extend(responses)
  response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

display(Markdown(response.choices[0].message.content))